                                                   # Chapter 9. Multimodal Large Language Models

In [ ]:
!pip install -q transformers==4.46.3 accelerate sentencepiece

In [ ]:
# It installs the Hugging Face transformers library version 4.46.3,
!pip install -q transformers==4.46.3

In [ ]:
#                   ==== OpenCLIP ====

In [ ]:
# imports the urlopen function from urllib.request module
# urlpen() = opens the given url and download its contents
# from urllib.request import urlopen

# PIL is a library used to work with images in Python.
# The Image class is used to open, create, edit, process, and save images in Python.
from PIL import Image

# Instead of using urllib to load the image from a URL (which gave an error),
# I uploaded the image to Colab and loaded it from the local path.
# Load an AI-generated image of a puppy playing in the snow
puppy_path = "/content/puppy.jpg"

# open the url form the given file path and converts each pixel into RGB colour format which is expected by model
image = Image.open(puppy_path).convert("RGB")

# create a caption to the image , later it will converted to embeddings and compares with the image embeddings
caption = "a puppy playing in the snow"


In [ ]:
#from transformers library importing
# CLIPModel = The CLIP model is used to understand both images and text together.
   # It converts both into embeddings (vectors) in the same vector space so they can be compared.
# CLIPTokenizerFast = tokenizer for the CLIP model , Converts text into token IDs
# CLIPProcessor = Preprocesses images and/or text into the format expected by the CLIP model (pixel values and input IDs).
from transformers import CLIPTokenizerFast, CLIPProcessor, CLIPModel

# This is the model we want to use
model_id = "openai/clip-vit-base-patch32"

# Load a tokenizer to preprocess the text
clip_tokenizer = CLIPTokenizerFast.from_pretrained(model_id)

# Load a processor to preprocess the images
clip_processor = CLIPProcessor.from_pretrained(model_id)

# Main model for generating text and image embeddings
model = CLIPModel.from_pretrained(model_id)

In [ ]:
# Tokenize the caption and retrun the result in pytorch tensor values
inputs = clip_tokenizer(caption, return_tensors="pt")
inputs

In [ ]:
# Convert our input back to tokens
  # Now we can understand what each ID represents.
clip_tokenizer.convert_ids_to_tokens(inputs["input_ids"][0])

In [ ]:
# Generate a 512-dimensional text embedding from the tokenized input using the CLIP text encoder.
# **inputs unpacks the input dictionary (input_ids and attention_mask) and passes them to the model.
text_embedding = model.get_text_features(**inputs)
print(text_embedding)

In [ ]:
# Preprocess image
processed_image = clip_processor(

  # No text input, only the image will be processed.
text=None,

    # Pass the input image to the processor.
images=image,

   # Return the processed image as a PyTorch tensor
  return_tensors="pt"

  # Extract only the processed image tensor (pixel_values) from the returned dictionary.
)["pixel_values"]

# Display the shape of the processed image tensor.
processed_image.shape


In [ ]:
# Visualizing the processed image

In [ ]:
# Import PyTorch library for tensor operations.
import torch

# Import NumPy for array operations.
import numpy as np

#  Import Matplotlib to display the image.
import matplotlib.pyplot as plt

# Prepare image for visualization
# Remove the batch dimension from the processed image.
# Shape changes from [1, 3, 224, 224] → [3, 224, 224].
img = processed_image.squeeze(0)

# Converts the image from [Channels, Height, Width] to [Width, Height, Channels].
img = img.permute(*torch.arange(img.ndim - 1, -1, -1))

# Swap the first two dimensions.
# Converts [Width, Height, Channels] → [Height, Width, Channels]
img = np.einsum("ijk->jik", img)

# Visualize preprocessed image
plt.imshow(img)

# Hide the x-axis and y-axis values.
plt.axis("off")

In [ ]:
# Generate a 512-dimensional image embedding from the preprocessed image using the CLIP image encoder
image_embedding = model.get_image_features(processed_image)
processed_image.shape



In [ ]:
# Extract the pooled (final) text embedding representing the entire input sentence.
# When using CLIPTextModel or CLIPVisionModel, the model returns multiple outputs,
# so we use .pooler_output to extract the single embedding that represents the entire text or image.
text_embedding = text_embedding.pooler_output

image_embedding = image_embedding.pooler_output

In [ ]:
# Normalize the text embedding by dividing each embedding value by the vector's magnitude (L2 norm)
text_embedding /= text_embedding.norm(dim=-1, keepdim=True)

# Normalize the image embedding by dividing each embedding value by the vector's magnitude (L2 norm).
image_embedding /= image_embedding.norm(dim=-1, keepdim=True)

# Stop gradient tracking, move the text embedding to the CPU, and convert it to a NumPy array.
text_embedding = text_embedding.detach().cpu().numpy()

# Stop gradient tracking, move the image embedding to the CPU, and convert it to a NumPy array.
image_embedding = image_embedding.detach().cpu().numpy()

# Calculate the similarity score by taking the dot product of the normalized text and image embeddings.
score = np.dot(text_embedding, image_embedding.T)

score

In [ ]:
from PIL import Image

# Load the images and convert it to RGB format
puppy = Image.open("/content/puppy.jpg").convert("RGB")
cat = Image.open("/content/cat.jpg").convert("RGB")
car = Image.open("/content/car.jpg").convert("RGB")

# Store all the images in a list.
images=[puppy,cat,car]

# Store the corresponding captions for each image.
captions=["a puppy playing in the snow"
"a cute cat"
"car on the road with sunset background"]

In [ ]:
#              ==== USING SENTENCE-TRANSFORMERS TO LOAD CLIP ====

In [ ]:
# Import the SentenceTransformer class to load pretrained embedding models,
# and util to compute cosine similarity between embeddings.
from sentence_transformers import SentenceTransformer, util

# Load the CLIP model that is compatible with the Sentence Transformers library.
model = SentenceTransformer("clip-ViT-B-32")

# Generate embeddings for all the input images.
image_embeddings = model.encode(images)

# Generate embeddings for all the input text captions.
text_embeddings = model.encode(captions)

# Compute the cosine similarity between every image embedding
# and every text embedding, resulting in a similarity matrix.
sim_matrix = util.cos_sim(
    image_embeddings,
    text_embeddings
)

# Display the similarity matrix.
print(sim_matrix)

In [ ]:
#                   Making Text Generation Models Multimodal
#                   Preprocessing Multimodal Inputs

In [ ]:
# Import AutoProcessor to preprocess both images and text,
# and Blip2ForConditionalGeneration to load the pretrained BLIP-2 model.
from transformers import AutoProcessor, Blip2ForConditionalGeneration

# Import PyTorch for tensor operations and GPU support.
import torch


# Load the BLIP-2 processor, which preprocesses images and text
# into the format expected by the model.
blip_processor = AutoProcessor.from_pretrained(
    "Salesforce/blip2-opt-2.7b"
)

# Load the pretrained BLIP-2 model for image captioning
# and visual question answering using 16-bit floating-point precision.
model = Blip2ForConditionalGeneration.from_pretrained(
    "Salesforce/blip2-opt-2.7b",
    torch_dtype=torch.float16
)

# Select GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"

# Move model to selected device
model.to(device)

In [ ]:
# Preprocessing images

In [ ]:
# Import urlopen to download and open an image directly from an online URL.
from urllib.request import urlopen

# Import the Image class from the Pillow (PIL) library to open and process images.
from PIL import Image

# Load image of a supercar
car_path = "https://raw.githubusercontent.com/HandsOnLLM/Hands-On-Large-Language-Models/main/chapter09/images/car.png"

# Download the image from the URL, open it, and convert it to RGB format.
image = Image.open(urlopen(car_path)).convert("RGB")

# Display the loaded image
image

In [ ]:
# Preprocess the input image and convert it into PyTorch tensors
inputs = blip_processor(image, return_tensors="pt").to(device,
torch.float16)

# Display the shape of the processed image tensor (pixel values)
inputs["pixel_values"].shape


In [ ]:
# Preprocessing text

In [ ]:
# Display the tokenizer used internally by the BLIP-2 processor for processing text.
blip_processor.tokenizer

In [ ]:
# Text preprocessing
text = "Her vocalization was remarkably melodic"

# Tokenize text
token_ids = blip_processor.tokenizer(
    text,
    return_tensors="pt"
)["input_ids"][0]


# Convert input IDs to tokens
tokens = blip_processor.tokenizer.convert_ids_to_tokens(
    token_ids
)

print(tokens)

In [ ]:
# Replace the space token with an underscore
tokens = [token.replace("Ġ", "_")
for token in tokens]
tokens

In [ ]:
#             ==== IMAGE CAPTIONING ====

In [ ]:
# Load the image from the given URL and convert it into RGB format
# RGB conversion ensures the image has 3 color channels required by the model
image = Image.open(urlopen(car_path)).convert("RGB")


# Preprocess the image and text input using BLIP-2 processor
# text="" means we are not providing any question or prompt.
# The model will generate a caption based only on the image.
inputs = blip_processor(
    images=image,
    text="",
    return_tensors="pt"
)


# Move all input tensors (pixel_values, input_ids, attention_mask)
# to the same device as the model (GPU if available, otherwise CPU)
inputs = {
    key: value.to(device)
    for key, value in inputs.items()
}


# Convert image pixel values to float16 to reduce memory usage
# and improve inference speed on GPUs.
# Keep input_ids and attention_mask as integer tensors.
inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)


# Generate output token IDs using the BLIP-2 model
# The image is processed by the vision encoder,
# converted into visual embeddings, and passed to the LLM
# to generate a caption.
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=20
    )


# Convert the generated token IDs into readable text
# skip_special_tokens=True removes tokens like <s>, </s>, etc.
generated_text = blip_processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0].strip()


# Display the generated image caption
print(generated_text)

In [ ]:
 #          ==== Multimodal Chat-Based Prompting ====

In [ ]:
# Create a chat-style prompt containing previous questions and answers.
prompt = """Question: Write down what you see in this picture.
Answer: A sports car driving on the road at sunset.
Question: What would it cost me to drive that car? Answer:"""

# Preprocess the input image and prompt, and convert them into PyTorch tensors.
inputs = blip_processor(image,
                        text=prompt,
                        return_tensors="pt"
                        ).to(device, torch.float16)

# Generate output token IDs from the image and text prompt.
# max_new_tokens=30 limits the response to a maximum of 30 new tokens.
generated_ids = model.generate(**inputs, max_new_tokens=30)


# Convert the generated token IDs into readable text and remove special tokens
generated_text = blip_processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

# Extract the generated response from the list and remove leading/trailing spaces.
generated_text = generated_text[0].strip()
generated_text


In [33]:
# Import HTML and display functions to show formatted text in the Jupyter notebook
from IPython.display import HTML, display

# Import ipywidgets to create interactive widgets such as text boxes and buttons.
import ipywidgets as widgets

# Define a function that is called whenever the user enters text in the input box
def text_eventhandler(*args):

  # Get the text entered by the user from the widget event
  question = args[0]["new"]

 # If the user entered a question,
  if question:

     # Clear the input text box after storing the question.
    args[0]["owner"].value = ""

In [ ]:
# Import HTML and display functions to show formatted text in the Jupyter notebook.
from IPython.display import HTML, display

# Import ipywidgets to create interactive widgets such as text boxes.
import ipywidgets as widgets


# Define a function that is called whenever the user enters text in the input box.
def text_eventhandler(*args):

    # Get the question entered by the user.
    question = args[0]["new"]

    # Check if the user entered a question.
    if question:

        # Clear the input box after storing the question.
        args[0]["owner"].value = ""

        # Create the prompt for the model.
        if not memory:
            # For the first question, create a simple prompt.
            prompt = "Question: " + question + " Answer:"
        else:
            # Template for storing previous questions and answers.
            template = "Question: {} Answer: {}."

            # Combine the conversation history with the current question.
            prompt = " ".join(
                [
                    template.format(memory[i][0], memory[i][1])
                    for i in range(len(memory))
                ]
            ) + " Question: " + question + " Answer:"

        # Preprocess the image and prompt into tensors.
        inputs = blip_processor(
            image,
            text=prompt,
            return_tensors="pt"
        )

        # Move the inputs to the selected device (CPU/GPU) using float16 precision.
        inputs = inputs.to(device, torch.float16)

        # Generate output token IDs from the image and prompt.
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=100
        )

        # Convert the generated token IDs into readable text.
        generated_text = blip_processor.batch_decode(
            generated_ids,
            skip_special_tokens=True
        )

        # Extract the generated answer and remove extra spaces.
        generated_text = generated_text[0].strip().split("Question")[0]

        # Store the current question and generated answer in memory.
        memory.append((question, generated_text))

        # Display the user's question.
        output.append_display_data(
            HTML("<b>USER:</b> " + question)
        )

        # Display the BLIP-2 generated answer.
        output.append_display_data(
            HTML("<b>BLIP-2:</b> " + generated_text)
        )

        # Add a blank line between conversations.
        output.append_display_data(
            HTML("<br>")
        )


# Create a text input box for the user.
in_text = widgets.Text()

# Trigger the event handler only after the user finishes typing.
in_text.continuous_update = False

# Call text_eventhandler() whenever the value of the text box changes.
in_text.observe(
    text_eventhandler,
    "value"
)

# Create an output area to display the conversation.
output = widgets.Output()

# Create an empty list to store previous questions and answers.
memory = []

# Display the chatbot interface.
display(
    widgets.VBox(
        # Place the output area and input box inside a vertical container.
        children=[
            output,
            in_text
        ],
        # Display the newest messages at the bottom like a chat application.
        layout=widgets.Layout(
            display="inline-flex",
            flex_flow="column-reverse"
        )
    )
)